## Deploy with vLLM

In [1]:
!pip install vllm openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 MB 19.3 MB/s  0:00:14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 91.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 MB 76.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.0/556.0 kB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 107.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 105.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 60.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 MB 42.9 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 37.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 38.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 32.0 MB/s  0:00:00
   ━━━━━━━━━

In [2]:
import vllm, torch
print(vllm.__version__, torch.cuda.get_device_name(0))

0.33.0 Tesla T4


Login to Hugging Face

In [1]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

start an OpenAI-compatible server, query it with the standard openai client, rerun your evaluation through the API, and measure speed. Copy SYSTEM, build_messages, clean, normalize, is_valid and score from Notebook A, and recreate the test split with the Step 1 code (same seed, same 200 rows).
1. Start the server in the background. --dtype half is required on the T4.

In [3]:
MODEL = "harimanikkothu/qwen2.5-1.5b-sql-merged"
!nohup vllm serve {MODEL} --dtype half --max-model-len 1024 --gpu-memory-utilization 0.85 > vllm.log 2>&1 &

In [4]:
import time, requests
for _ in range(60):
    try:
        if requests.get("http://localhost:8000/v1/models").ok:
            print("server ready"); break
    except requests.ConnectionError:
        pass
    time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])   # show the error

server ready


DataSet

In [5]:
from datasets import load_dataset
import re, sqlite3

# Same split as Notebook A
ds = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=42)
test_ds = ds.select(range(200))
test_examples = list(test_ds)

# Same prompt format as training
SYSTEM = ("You are a SQL expert. Given a table schema and a question, "
          "write one SQLite query that answers it. Return only the SQL.")

def build_messages(ex, with_answer=True):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Schema:\n{ex['context']}\n\nQuestion: {ex['question']}"},
    ]
    if with_answer:
        msgs.append({"role": "assistant", "content": ex["answer"]})
    return msgs

# Same scoring functions
def clean(text):
    m = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def normalize(sql):
    return " ".join(sql.strip().rstrip(";").lower().split())

def is_valid(sql, schema):
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(schema)
        con.execute(sql)
        return True
    except Exception:
        return False
    finally:
        con.close()

def score(preds, examples):
    sqls = [clean(p) for p in preds]
    n = len(examples)
    return {
        "exact_match": sum(normalize(s) == normalize(e["answer"]) for s, e in zip(sqls, examples)) / n,
        "valid_sql":   sum(is_valid(s, e["context"]) for s, e in zip(sqls, examples)) / n,
    }

print(test_examples[0]["question"])

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

When Essendon played away; where did they play?


2. Call it like any OpenAI-style API. This is why vLLM is popular: apps written for hosted APIs (including LangChain) work against your own model by changing one URL.

In [7]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

def ask(ex, model=MODEL):
    r = client.chat.completions.create(
        model=model, messages=build_messages(ex, with_answer=False),
        temperature=0, max_tokens=128)
    return r.choices[0].message.content

print(test_examples[0])
print(ask(test_examples[0]))

{'answer': 'SELECT venue FROM table_name_50 WHERE away_team = "essendon"', 'question': 'When Essendon played away; where did they play?', 'context': 'CREATE TABLE table_name_50 (venue VARCHAR, away_team VARCHAR)'}
SELECT venue FROM table_name_50 WHERE away_team = "essendon"


3. Rerun the evaluation through the server. Scores should be close to Step 4. Small differences are expected: you trained against a 4-bit base, but the merged model runs in 16-bit. Note the difference in your README.

In [8]:
api_preds = [ask(ex) for ex in test_examples]
print(score(api_preds, test_examples))

{'exact_match': 0.7, 'valid_sql': 0.96}


4. Measure speed under load. Send 200 requests with 16 in flight at once, the way real users would hit it:

In [9]:
import statistics
from concurrent.futures import ThreadPoolExecutor

def timed(ex):
    t = time.perf_counter()
    r = client.chat.completions.create(model=MODEL,
        messages=build_messages(ex, with_answer=False), temperature=0, max_tokens=128)
    return time.perf_counter() - t, r.usage.completion_tokens

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=16) as pool:
    results = list(pool.map(timed, test_examples))
total = time.perf_counter() - start

lat = sorted(r[0] for r in results)
print(f"requests/s: {len(results)/total:.1f}")
print(f"output tokens/s: {sum(r[1] for r in results)/total:.0f}")
print(f"p50 latency: {statistics.median(lat):.2f}s   p95: {lat[int(0.95*len(lat))]:.2f}s")

requests/s: 27.0
output tokens/s: 710
p50 latency: 0.55s   p95: 0.87s


with max_worker = 1

In [10]:
import statistics
from concurrent.futures import ThreadPoolExecutor

def timed(ex):
    t = time.perf_counter()
    r = client.chat.completions.create(model=MODEL,
        messages=build_messages(ex, with_answer=False), temperature=0, max_tokens=128)
    return time.perf_counter() - t, r.usage.completion_tokens

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=1) as pool:
    results = list(pool.map(timed, test_examples))
total = time.perf_counter() - start

lat = sorted(r[0] for r in results)
print(f"requests/s: {len(results)/total:.1f}")
print(f"output tokens/s: {sum(r[1] for r in results)/total:.0f}")
print(f"p50 latency: {statistics.median(lat):.2f}s   p95: {lat[int(0.95*len(lat))]:.2f}s")

requests/s: 2.1
output tokens/s: 55
p50 latency: 0.47s   p95: 0.72s


5. Bonus: serve the adapter instead of a merged model. vLLM can load the base model once and attach LoRA adapters per request. One server then hosts many fine-tunes cheaply.

In [ ]:
!pkill -f "vllm serve"; sleep 5
from huggingface_hub import snapshot_download
lora_path = snapshot_download("harimanikkothu/qwen2.5-1.5b-sql-lora")

!nohup vllm serve Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 1024 --enable-lora --max-lora-rank 16 --lora-modules sql={lora_path} > vllm.log 2>&1 &